# NB03: Feature extraction (OpenCLIP ViT-B/16, LAION-2B)

Extracts tile-level embeddings for every TCGA WSI and aggregates to the
patient level via mean pooling.

**Manuscript reference** (Methods, Supp Table S6):
- Backbone: OpenCLIP ViT-B/16, pretrained `laion2b_s34b_b88k` (512-dimensional)
- Preprocessing: GPU resize to 224 px + CLIP mean/std normalization
- Hardware: NVIDIA RTX 4090 (24 GB VRAM); ~143.7 hours total for 19,996 slides
- Throughput: 139.2 slides/hour, ~53 tiles/sec
- Aggregation: patient-level mean pooling across all tiles from all diagnostic slides

**Required env**: `WSI_ROOT`, `WORKSPACE`.
**Inputs**: `coords/{slide_id}.csv` from NB02.
**Outputs**: `embeddings/{slide_id}.npy` per slide; `embeddings/patient_means.parquet`;
`embeddings/per_site_emb_stats.parquet` (used by NB10 for Supp Table S10 / Fig 4D).

In [ ]:
import os
import re
import time
import json
import gc
import hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import openslide
import torch
import torch.nn.functional as F
import open_clip

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
WSI_ROOT = Path(os.environ["WSI_ROOT"]).resolve()
COORDS_DIR = WORKSPACE / "coords"
EMB_DIR = WORKSPACE / "embeddings"
RUNS_DIR = WORKSPACE / "runs"
EMB_DIR.mkdir(parents=True, exist_ok=True)

# inference params
BATCH_SIZE = 256          # auto-halved on OOM
USE_AMP = True
SAVE_DTYPE = "float16"
TARGET_SZ = 224
PREFETCH_THREADS = 2
WSI_EXTS = (".svs", ".tif", ".tiff", ".ndpi", ".mrxs", ".scn", ".bif", ".svslide")

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEV}")


In [ ]:
# load OpenCLIP ViT-B/16 (laion2b_s34b_b88k); manuscript-locked backbone
model, _preprocess, _ = open_clip.create_model_and_transforms(
    "ViT-B-16", pretrained="laion2b_s34b_b88k", device=DEV
)
model.eval()
FEAT_DIM = getattr(model.visual, "output_dim", 512)
assert FEAT_DIM == 512, f"unexpected feature dim {FEAT_DIM}; manuscript reports 512-d"

# weights provenance hash for reproducibility audit
def weights_sha256(mod):
    h = hashlib.sha256()
    sd = mod.state_dict()
    for k in sorted(sd.keys()):
        t = sd[k].detach().cpu().contiguous()
        h.update(k.encode("utf-8"))
        h.update(t.numpy().tobytes(order="C"))
    return h.hexdigest()

WEIGHTS_HASH = weights_sha256(model)
print(f"feature dim    : {FEAT_DIM}")
print(f"weights sha256 : {WEIGHTS_HASH}")


In [ ]:
# CLIP mean/std on GPU; preprocessing inline for speed
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=DEV).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=DEV).view(1, 3, 1, 1)

@torch.inference_mode()
def preprocess_batch(imgs):
    """imgs: list of uint8 HxWx3 arrays. returns float tensor [N,3,224,224] normalized."""
    arr = np.stack(imgs, axis=0)
    t = torch.from_numpy(arr).to(DEV, non_blocking=True).permute(0, 3, 1, 2).float() / 255.0
    t = F.interpolate(t, size=(TARGET_SZ, TARGET_SZ), mode="bilinear", align_corners=False)
    return (t - CLIP_MEAN) / CLIP_STD

@torch.inference_mode()
def encode(t):
    with torch.amp.autocast("cuda", enabled=USE_AMP):
        z = model.encode_image(t)
    if z.ndim > 2:
        z = z.mean(dim=(2, 3))
    return z.float()


In [ ]:
# build slide index and patient mapping
def index_wsis(root):
    return {p.stem: p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in WSI_EXTS}

def patient_id(slide_id):
    m = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", slide_id, re.I)
    return m.group(1).upper() if m else slide_id

def site_code(patient):
    m = re.match(r"^TCGA-([A-Z0-9]{2})-", patient)
    return m.group(1) if m else "UNK"

wsi_idx = index_wsis(WSI_ROOT)
coord_files = sorted(COORDS_DIR.glob("*.csv"))
todo = [c.stem for c in coord_files if c.stem in wsi_idx]
print(f"slides with coords + WSI: {len(todo):,}")


In [ ]:
def read_batch(slide, coords_df, a, b):
    imgs = []
    for k in range(a, b):
        x, y, w, h, lvl = coords_df.iloc[k][["x", "y", "w", "h", "level"]]
        region = slide.read_region((int(x), int(y)), int(lvl), (int(w), int(h))).convert("RGB")
        imgs.append(np.array(region, dtype=np.uint8))
    return imgs

def encode_slide(slide_path, coord_csv, batch_size=BATCH_SIZE):
    coords = pd.read_csv(coord_csv)
    n = len(coords)
    if n == 0:
        return np.zeros((0, FEAT_DIM), dtype=np.float16 if SAVE_DTYPE == "float16" else np.float32), 0.0

    slide = openslide.OpenSlide(str(slide_path))
    feats = []
    bs = batch_size
    t0 = time.time()
    try:
        with ThreadPoolExecutor(max_workers=PREFETCH_THREADS) as pool:
            i = 0
            j = min(n, bs)
            fut = pool.submit(read_batch, slide, coords, i, j)
            i = j
            while True:
                imgs = fut.result()
                if i < n:
                    j = min(n, i + bs)
                    fut = pool.submit(read_batch, slide, coords, i, j)
                    i = j
                    has_next = True
                else:
                    has_next = False
                t = preprocess_batch(imgs)
                z = encode(t)
                feats.append(z)
                if not has_next:
                    break
    finally:
        slide.close()

    emb = torch.cat(feats, dim=0).cpu().numpy()
    if SAVE_DTYPE == "float16":
        emb = emb.astype(np.float16, copy=False)
    return emb, time.time() - t0


In [ ]:
# main loop with OOM retry and resume-safe per-slide outputs
RUN_DIR = RUNS_DIR / f"emb_{int(time.time())}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
log_path = RUN_DIR / "run.log"

def log(msg):
    ts = time.strftime("[%Y-%m-%d %H:%M:%S]")
    with open(log_path, "a") as f:
        f.write(f"{ts} {msg}\n")
    print(msg)

per_site = {}
failed = []
done = 0
t0_all = time.time()

for sid in todo:
    out_npy = EMB_DIR / f"{sid}.npy"
    if out_npy.exists():
        done += 1
        continue
    coord_csv = COORDS_DIR / f"{sid}.csv"
    slide_path = wsi_idx[sid]
    bs = BATCH_SIZE

    while True:
        try:
            emb, sec = encode_slide(slide_path, coord_csv, batch_size=bs)
            np.save(out_npy, emb)
            log(f"OK {sid} tiles={emb.shape[0]} sec={sec:.1f}")
            site = site_code(patient_id(sid))
            s = per_site.setdefault(site, {"slides": 0, "norm_sum": 0.0, "tiles": 0})
            s["slides"] += 1
            s["tiles"] += emb.shape[0]
            if emb.size:
                s["norm_sum"] += float(np.linalg.norm(emb.astype(np.float32), axis=1).mean())
            done += 1
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= 32:
                log(f"OOM_FAIL {sid}")
                failed.append(sid)
                break
            bs = max(32, bs // 2)
            log(f"OOM_RETRY {sid} new_bs={bs}")
        except Exception as e:
            log(f"ERROR {sid} {repr(e)}")
            failed.append(sid)
            break
    gc.collect()
    torch.cuda.empty_cache()

elapsed_h = (time.time() - t0_all) / 3600
log(f"done: {done} processed, {len(failed)} failed, elapsed {elapsed_h:.2f} h")


In [ ]:
# aggregate to patient means and write parquet
def patient_mean_from_slides(slide_ids):
    acc = np.zeros(FEAT_DIM, dtype=np.float64)
    n_tiles = 0
    for sid in slide_ids:
        f = EMB_DIR / f"{sid}.npy"
        if not f.exists():
            continue
        arr = np.load(f, mmap_mode="r")
        if arr.size == 0:
            continue
        acc += arr.astype(np.float64).sum(axis=0)
        n_tiles += arr.shape[0]
    return (acc / n_tiles).astype(np.float32) if n_tiles > 0 else None

slide_to_patient = {sid: patient_id(sid) for sid in todo}
patient_to_slides = {}
for sid, pid in slide_to_patient.items():
    patient_to_slides.setdefault(pid, []).append(sid)

print(f"unique patients: {len(patient_to_slides):,}")

rows = []
for pid, sids in patient_to_slides.items():
    mu = patient_mean_from_slides(sids)
    if mu is None:
        continue
    rows.append([pid] + mu.tolist())

cols = ["patient"] + [f"f{i:03d}" for i in range(FEAT_DIM)]
patient_means = pd.DataFrame(rows, columns=cols)
out_parquet = EMB_DIR / "patient_means.parquet"
patient_means.to_parquet(out_parquet, index=False)
print(f"patient means: {patient_means.shape} -> {out_parquet}")


In [ ]:
# per-site (TSS) embedding norm summary; consumed by NB10 for Supp S10 / Fig 4D
site_rows = []
for site, s in per_site.items():
    site_rows.append({
        "site": site,
        "slides": int(s["slides"]),
        "tiles": int(s["tiles"]),
        "mean_norm": float(s["norm_sum"] / max(1, s["slides"])),
    })
pd.DataFrame(site_rows).to_parquet(EMB_DIR / "per_site_emb_stats.parquet", index=False)

print(f"unique TSS sites encountered: {len(site_rows)}")
print(f"manuscript ref: 710 TSS across the full TCGA cohort (Supp Table S10)")

report = {
    "backbone": "openclip_vitb16_laion2b_s34b_b88k",
    "feat_dim": int(FEAT_DIM),
    "weights_sha256": WEIGHTS_HASH,
    "amp": USE_AMP,
    "preprocess": "gpu_resize_224_clip_meanstd",
    "slides_embedded": int(done),
    "failed": failed,
    "elapsed_hours": round((time.time() - t0_all) / 3600, 2),
    "patients": int(len(patient_to_slides)),
    "patient_means_path": str(out_parquet),
}
(RUN_DIR / "report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
